In [ ]:
import random as rnd
import numpy as np
import math
import matplotlib.pyplot as plt

In [ ]:
def generateInstace(n):
    lats = [rnd.random() * 10 for i in range(n)]
    lngs = [rnd.random() * 10 for i in range(n)]
    distance_matrix = np.zeros([n, n], dtype=float)
    for i in range(n):
        for j in range(n):
            distance_matrix[i, j] = math.sqrt((lats[j] - lats[i])**2 + (lngs[j] - lngs[i])**2)
    return [distance_matrix, lats, lngs]

def evaluate(pi, instance):
    last = pi[0]
    dist = 0.0
    for city in pi[1:]:
        dist += instance[0][last, city]
        last = city
    dist += instance[0][last, pi[0]]
    return dist

def plot_tsp_solution(pi, lats, lngs):
    plt.figure(figsize=(8, 6))
    for i in range(len(pi)):
        start_city = pi[i]
        end_city = pi[(i + 1) % len(pi)]
        plt.plot([lngs[start_city], lngs[end_city]], [lats[start_city], lats[end_city]], 'b-')

    plt.plot(lngs, lats, 'ro')
    for i, (x, y) in enumerate(zip(lngs, lats)):
        plt.text(x, y, str(i), color="black", fontsize=12)

    plt.title("TSP Solution Visualization")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.grid()
    plt.show()
n = 500
inst = generateInstace(n)
pi_rnd = list(range(n))
rnd.shuffle(pi_rnd)
plot_tsp_solution(pi_rnd, inst[1], inst[2])
print(evaluate(pi_rnd, inst))

In [ ]:
def insa(instance, n, pi = None):
    if pi is None:
        pi = [rnd.randint(0, n - 1)]

    while len(pi) < n:
        best_insert_city = None
        best_position = None
        best_cost = float('inf')

        unvisited = [city for city in range(n) if city not in pi]
        new_city = rnd.choice(unvisited)

        for i in range(len(pi) + 1):
            new_pi = pi[:i] + [new_city] + pi[i:]
            cost = evaluate(new_pi, instance)
            if cost < best_cost:
                best_cost = cost
                best_position = i
        pi.insert(best_position, new_city)

    return pi
pi_insa = insa(inst, n)
plot_tsp_solution(pi_insa, inst[1], inst[2])
print(evaluate(pi_insa, inst))

In [ ]:
def plot_ga_progress(avg_fit, gen_best, glob_best):
    plt.figure(figsize=(10, 6))
    plt.plot(avg_fit, label='Średni Fitness', color='blue', linestyle='--')
    plt.plot(gen_best, label='Najlepszy w Generacji', color='green')
    plt.plot(glob_best, label='Globalny Best', color='red', linewidth=2)
    plt.xlabel('Generacja')
    plt.ylabel('Wartość Funkcji Celu')
    plt.title('Postęp Algorytmu Genetycznego')
    plt.legend()
    plt.grid(True)
    plt.show()

def ga(instance, n, population_size=100, generations=1000, mutation_rate=0.01, verbose=True):
    population = [rnd.sample(range(n), n) for _ in range(population_size)]
    
    best_global = None
    best_fitness = float('inf')
    history_avg_fitness = []
    history_gen_best = []
    history_global_best = []

    for generation in range(generations):
        fitness = [evaluate(individual, instance) for individual in population]
        best_current = population[np.argmin(fitness)][:]
        fitness_current = fitness[np.argmin(fitness)]
        avg_fitness = sum(fitness) / population_size

        if fitness_current < best_fitness:
            best_fitness = fitness_current
            best_global = best_current[:]
            
        history_avg_fitness.append(avg_fitness)
        history_gen_best.append(fitness_current)
        history_global_best.append(best_fitness)
        
        if verbose and generation % 50 == 0:
            print(f"Generacja {generation}: Best = {best_fitness:.2f}, Avg = {avg_fitness:.2f}")

        new_population = []
        for i in range(0, population_size, 2):
            parent1, parent2 = rnd.sample(population, 2)
            crossover_point = rnd.randint(1, n - 1)
            child1 = parent1[:crossover_point] + [city for city in parent2 if city not in parent1[:crossover_point]]
            child2 = parent2[:crossover_point] + [city for city in parent1 if city not in parent2[:crossover_point]]

            if rnd.random() < mutation_rate:
                idx1, idx2 = rnd.sample(range(n), 2)
                child1[idx1], child1[idx2] = child1[idx2], child1[idx1]

            if rnd.random() < mutation_rate:
                idx1, idx2 = rnd.sample(range(n), 2)
                child2[idx1], child2[idx2] = child2[idx2], child2[idx1]

            new_population.extend([child1, child2])
        population = new_population
    if verbose:
        plot_ga_progress(history_avg_fitness, history_gen_best, history_global_best)
    return best_global
pi_ga = ga(inst, n, generations=1000)
plot_tsp_solution(pi_ga, inst[1], inst[2])
print(evaluate(pi_ga, inst))

In [ ]:
def ga2(instance, n, population_size=100, generations=1000, mutation_rate=0.01, verbose=True, selection="roulette", init_population="random"):
    population = [rnd.sample(range(n), n) for _ in range(population_size)]
    if init_population == "insa":
        for i in range(5):
            population[i] = insa(instance, n)

    best_global = None
    best_fitness = float('inf')
    history_avg_fitness = []
    history_gen_best = []
    history_global_best = []

    for generation in range(generations):
        fitness = [evaluate(individual, instance) for individual in population]
        best_current = population[np.argmin(fitness)][:]
        fitness_current = fitness[np.argmin(fitness)]
        avg_fitness = sum(fitness) / population_size

        if fitness_current < best_fitness:
            best_fitness = fitness_current
            best_global = best_current[:]
            
        history_avg_fitness.append(avg_fitness)
        history_gen_best.append(fitness_current)
        history_global_best.append(best_fitness)
        
        if verbose and generation % 50 == 0:
            print(f"Generacja {generation}: Best = {best_fitness:.2f}, Avg = {avg_fitness:.2f}")


        new_population = []
        sum_fitness = sum(fitness)
        for i in range(0, population_size, 2):

            
            if selection == "roulette":
                parent1, parent2 = rnd.choices(population, weights=[(1/f)/(1/sum_fitness) for f in fitness], k=2)
            if selection == "tournament":
                tournament_size = 3
                tournament_indices = rnd.sample(range(population_size), tournament_size)
                tournament_individuals = [population[idx] for idx in tournament_indices]
                tournament_fitness = [fitness[idx] for idx in tournament_indices]
                parent1 = tournament_individuals[np.argmin(tournament_fitness)]
                
                tournament_indices = rnd.sample(range(population_size), tournament_size)
                tournament_individuals = [population[idx] for idx in tournament_indices]
                tournament_fitness = [fitness[idx] for idx in tournament_indices]
                parent2 = tournament_individuals[np.argmin(tournament_fitness)]
            crossover_point = rnd.randint(1, n - 1)
            child1 = parent1[:crossover_point] + [city for city in parent2 if city not in parent1[:crossover_point]]
            child2 = parent2[:crossover_point] + [city for city in parent1 if city not in parent2[:crossover_point]]

            if rnd.random() < mutation_rate:
                idx1, idx2 = rnd.sample(range(n), 2)
                child1[idx1], child1[idx2] = child1[idx2], child1[idx1]

            if rnd.random() < mutation_rate:
                idx1, idx2 = rnd.sample(range(n), 2)
                child2[idx1], child2[idx2] = child2[idx2], child2[idx1]

            new_population.extend([child1, child2])
        population = new_population
    if verbose:
        plot_ga_progress(history_avg_fitness, history_gen_best, history_global_best)
    return best_global
pi_ga = ga2(inst, n, generations=1000, selection="tournament", init_population="insa", mutation_rate=0.20)
plot_tsp_solution(pi_ga, inst[1], inst[2])
print(evaluate(pi_ga, inst))

W klasycznym GA (tzw. Simple GA) operatory są losowe, co nie zawsze jest efektywne dla problemów takich jak TSP.

## Inicjalizacja Populacji Początkowej
Zamiast pełnej losowości, można „podpowiedzieć” algorytmowi kierunek poszukiwań:

 - Heuristic Initialization: Wstawienie do populacji kilku osobników wygenerowanych algorytmem zachłannym (np. INSA – Insertion Algorithm lub Nearest Neighbor). Dzięki temu startujemy z wyższego poziomu jakości, co przyspiesza zbieżność.

 - Diversified Random: Generowanie osobników tak, aby maksymalnie pokryć przestrzeń poszukiwań (np. sekwencje Sobola), co zapobiega zbyt szybkiej dominacji jednego genotypu.

## Mechanizmy Selekcji
Decydują o tym, które geny przetrwają

 - Roulette Wheel - szansa na wybór jest proporcjonalna do dopasowania (Wysokie ryzyko przedwczesnej zbieżności przez „super-osobniki”).
 - Tournament - Wybieramy $k$ osobników i wygrywa najlepszy z nich (Łatwa kontrola presji selekcyjnej poprzez zmianę rozmiaru turnieju $k$).
 - Rank Selection - Sortujemy populację i przypisujemy wagę na podstawie miejsca w rankingu (Niweluje różnice skali w fitness, utrzymując stałą presję).

 ## Algorytmy Wyspowe
 To podejście do równoległego algorytmu genetycznego, które świetnie nadaje się do tłumaczenia koncepcji ewolucji rozproszonej.

 - Populacja jest dzielona na kilka mniejszych podpopulacji („wysp”).
 - Każda wyspa ewoluuje niezależnie przez określony czas.
 - Co $X$ generacji najlepsze osobniki „przepływają” (są kopiowane) z jednej wyspy na drugą.
 - Pozwala uniknąć lokalnych minimów – każda wyspa może eksplorować inny obszar przestrzeni, a migracja wprowadza „świeżą krew”.

 ## Zaawansowane Operatory Krzyżowania
 W problemie komiwojażera zwykłe krzyżowanie jednegopunktowe może tworzyć nieprawidłowe.

 - OX (Order Crossover) zachowuje względną kolejność miast z jednego rodzica, uzupełniając resztę od drugiego.
 - PMX (Partially Mapped Crossover) mapuje fragmenty trasy, dbając o to, by każde miasto wystąpiło dokładnie raz.
 - Edge Recombination, skupia się na zachowaniu krawędzi (połączeń między miastami) zamiast ich pozycji w tablicy.

## Strategie Zastępowania
 - Generational, nowa populacja całkowicie zastępuje starą.
 - Steady-State, w każdej generacji wymieniamy tylko 1-2 najgorsze osobniki na nowe dzieci.
 - Elitism,  gwarantujemy, że $E$ najlepszych osobników z generacji $t$ przejdzie bez zmian do $t+1$, zapobiega to przypadkowej utracie najlepszego znalezionego rozwiązania.

# Jednomazynowy problem szeregowana zadań z ważoną sumą opóźnień

Problem $1|w_j,d_j|\sum w_jT_j$ możemy zapisać w skrócie $1||\sum w_jT_j$, ponieważ parametry problemu wynikają z kryterium optymalizacyjnego. W Problemie mamy zbiór \mathcal{J}=\{1,2,\dots,n\}, składający się z $n$~zadań wykonywanych na maszynie. każde $j$-te zadanie składa się z trzech parametrów:
- $p_j$ - czas wykonania (performed time),
- $w_j$ - waga (weight)/współczynnik kary,
- $d_j$ - żądany termin zakończenia (deadline).

Każde zadanie  musi być wykonywane nieprzerwanie przez $p_{\pi_(j)}$ czasu na maszynie. Naraz może wykonywać się tylko jedno zadanie. Każde zadanie powinno być ukończone przed jego żądanym terminem ukończenia $d_{\pi_(j)}$, w przeciwnym wypadku zostanie naliczona kara. Naszym zadaniem jest ustalenie  harmonogramu. Zadania nie wymagają przygotowania, tzn. są wszystkie odrazu dostępne, więc możemy zacząć od razu wykonywać pierwsze zdanie: $S_{\pi(1)} = 0$ . W omawianym problemie zachowujemy ciągłość pracy maszyny, w zwiazku z czym  moment rozpoczęcia kolejnego zadania jest zawsze momentem zakończenia poprzedniego $ S_{\pi(j)} = C_{\pi(j-1)}$. Dla każdego zadania należy obliczyć jego spóźnienie $T_{\pi_(j)}\geq0$, pamiętając że nie premiujemy wcześniejszego wykonania zadania, więc wartośc nie może być ujemna $T_{\pi(j)}=max(C_{\pi(j)}-d_{\pi(j)},0)$. Badanym kryterium optymalizacyjnym jest ważona suma  opóźnień wszystkich zadań:

 $F(\pi) = \sum\limits^n_{j\in\mathcal{J}} w_{\pi(j)}T_{\pi(j)}$.

 Szukamy takiego uszeregowania $\pi$ aby wartość funkcji $F(\pi)$ byłą jak najmniejsza.

In [ ]:
def generateInstace(n):
    p = [rnd.randint(1, 29) for i in range(n)]
    w = [rnd.randint(1, 5) for i in range(n)]
    X = sum(p)
    d = [rnd.randint(1, X) for i in range(n)]
    return [p, w, d]

def evaluate(pi, instance):
    p, w, d = instance
    current_time = 0
    total_weighted_tardiness = 0
    for task_idx in pi:
        current_time += p[task_idx]
        tardiness = max(0, current_time - d[task_idx])
        total_weighted_tardiness += w[task_idx] * tardiness
        
    return total_weighted_tardiness

def plot_witi_solution(pi, instance):
    p, w, d = instance
    n = len(pi)
    
    fig, ax = plt.subplots(figsize=(12, 4))
    current_time = 0
    
    for task_idx in pi:
        duration = p[task_idx]
        due_date = d[task_idx]
        
        # Sprawdzenie czy zadanie będzie opóźnione
        finish_time = current_time + duration
        is_tardy = finish_time > due_date
        color = 'salmon' if is_tardy else 'skyblue'
        
        # Rysowanie bloku zadania
        ax.broken_barh([(current_time, duration)], (10, 9), facecolors=color, edgecolor='black')
        
        # Tekst z numerem zadania i terminem (d)
        mid_x = current_time + duration/2
        ax.text(mid_x, 14.5, f"Z{task_idx}", ha='center', va='center', fontweight='bold')
        ax.text(mid_x, 11.5, f"d:{due_date}", ha='center', va='center', fontsize=8)
        
        current_time = finish_time

    ax.set_ylim(5, 25)
    ax.set_xlim(0, sum(p))
    ax.set_xlabel('Czas')
    ax.set_yticks([])
    ax.set_title(f'Harmonogram WiTi (Suma ważonych opóźnień: {evaluate(pi, instance)})')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()



n = 20
inst_witi = generateInstace(n)
print("Instancja WiTi:")
print("p:", inst_witi[0])
print("w:", inst_witi[1])   
print("d:", inst_witi[2])
pi_rnd = list(range(n))
rnd.shuffle(pi_rnd)
print("Losowe rozwiązanie:", pi_rnd)
plot_witi_solution(pi_rnd, inst_witi)
print(evaluate(pi_rnd, inst_witi))


In [ ]:
pi_insa = insa(inst_witi, n)
print("Rozwiązanie INSA:", pi_insa)
plot_witi_solution(pi_insa, inst_witi)
print(evaluate(pi_insa, inst_witi))


In [ ]:
def greedy_witi(instance):
    p, w, d = instance
    n = len(p)
    tasks = list(range(n))
    tasks.sort(key=lambda i: (d[i], -w[i]/p[i]))
    return tasks

pr_geedy = greedy_witi(inst_witi)
print("Rozwiązanie Greedy:", pr_geedy)
plot_witi_solution(pr_geedy, inst_witi)
print(evaluate(pr_geedy, inst_witi))

In [ ]:
pi_ga = ga2(inst_witi, n, generations=1000, selection="tournament", init_population="insa")
plot_witi_solution(pi_ga, inst_witi)
print(evaluate(pi_ga, inst_witi))